In [1]:
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

from tensorflow.keras.applications import (
    EfficientNetB0, EfficientNetB3, EfficientNetB5,
    ResNet50, ResNet101, VGG16, VGG19, InceptionV3,
    DenseNet121, MobileNetV2, MobileNetV3Large
)

import albumentations as A
from sklearn.utils.class_weight import compute_class_weight
from utils import *

In [ ]:
# PATHS: Using just the local data for training, validation and testing.

TRAIN_PATH = Path(r"../data/dataset/train")
VAL_PATH = Path(r"../data/dataset/val")
TEST_PATH = Path(r"../data/dataset/test")
RESULTS_PATH = Path("../models")
RESULTS_PATH.mkdir(exist_ok=True)

TRAIN_CLASSES = sorted([d.name for d in TEST_PATH.iterdir() if d.is_dir()])
class_indices = {cls: i for i, cls in enumerate(TRAIN_CLASSES)}

CONF_MATRIX_DIR = RESULTS_PATH / "confusion_matrix"
CONF_MATRIX_DIR.mkdir(parents=True, exist_ok=True)

# If want to use the public images corresponding to common classes for both local and public (for the Transfer Learning experiment):

#TRAIN_PATH  = Path(r"../data/EXPERIMENT1/domain_adaptation/train")
#VAL_PATH  = Path(r"../data/EXPERIMENT1/domain_adaptation/val")

# PARAMS FOR TRANING
IMG_SIZE = [(160, 160), (224, 224), (299, 299)]
BATCH_SIZE = [16, 32, 64]
EPOCHS = [50, 100]
PATIENCE = [8, 10]

VAL_SPLIT = 0.2

# Custom augment:
CUSTOM_AUGMENTATION = []
for img_size in IMG_SIZE:
    custom_augmentation = A.Compose([
        A.Rotate(limit=10, border_mode=cv2.BORDER_CONSTANT, fill=0, p=0.5),
        A.RandomResizedCrop(
            size=(img_size[0], img_size[1]),
            scale=(0.9, 1.0),
            ratio=(0.9, 1.1),
            interpolation=cv2.INTER_LINEAR,
            p=0.3
        ),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.2),

        A.RandomBrightnessContrast(brightness_limit=0.08, contrast_limit=0.08, p=0.3),

        A.HueSaturationValue(hue_shift_limit=3, sat_shift_limit=5, val_shift_limit=4, p=0.2),

        A.ShiftScaleRotate(
            shift_limit=0.05,
            scale_limit=0.05,
            rotate_limit=0,
            border_mode=cv2.BORDER_CONSTANT,
            fill=0,
            p=0.3
        ),

        A.Resize(width=img_size[1], height=img_size[0])
    ], p=1.0)
    CUSTOM_AUGMENTATION.append(custom_augmentation)

In [ ]:
# backbone -> model from keras.
BACKBONES = {
    "resnet50": ResNet50,
    "resnet101": ResNet101,
    "vgg16": VGG16,
    "vgg19": VGG19,
    "inceptionv3": InceptionV3,
    "efficientnetb0": EfficientNetB0,
    "efficientnetb3": EfficientNetB3,
    "efficientnetb5": EfficientNetB5,
    "densenet121": DenseNet121,
    "mobilenetv2": MobileNetV2,
    "mobilenetv3large": MobileNetV3Large
}

def build_model(backbone_name, input_shape=(224,224,3), num_classes=6, weights="imagenet"):
    """
    backbone_name: str ("resnet50", "efficientnetb0", etc.)
    input_shape: tuple (H, W, 3)
    num_classes: number of output classes
    weights: name of the dataset of pre-learned weights for the backbone.
    """
    backbone_name = backbone_name.lower()
    if backbone_name not in BACKBONES:
        raise ValueError(f"Backbone {backbone_name} not supported. Options: {list(BACKBONES.keys())}")

    BackboneClass = BACKBONES[backbone_name]

    # Backbone
    base_model = BackboneClass(
        include_top=False,
        input_shape=input_shape,
        weights=weights,
        pooling=None
    )

    # head modified
    x = base_model.output
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Flatten()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    output = layers.Dense(num_classes, activation="softmax")(x)

    model = models.Model(inputs=base_model.input, outputs=output)

    return model, base_model

In [ ]:
model_list = [
    "mobilenetv2",
    "efficientnetb0",
    "mobilenetv3large",
    "densenet121",
    "efficientnetb3",
    "resnet50",
    "Inceptionv3",
    "resnet101",
    "vgg16",
    "vgg19",
    "efficientnetb5"
]

In [ ]:
def preprocess_for_backbone(img, backbone_name):
    """
    Apply the appropriate preprocessing to the image according to the model architecture.
    """
    X255 = img * 255.0
    b = backbone_name.lower()

    if "resnet" in b:
        return preprocess_resnet(X255)
    elif "vgg" in b:
        return preprocess_vgg16(X255)
    elif "inception" in b:
        return preprocess_inception(X255)
    elif "efficientnet" in b:
        return preprocess_effnet(X255)
    elif "densenet" in b:
        return preprocess_densenet(X255)
    elif "mobilenetv2" in b:
        return preprocess_mobilenetv2(X255)
    elif "mobilenetv3" in b:
        return preprocess_mobilenetv3(X255)
    else:
        print("ERROR:")
        return None

In [ ]:
for backbone_name in model_list:
    print(f"\n=== Training model: {backbone_name} ===")

    # data gen
    train_gen = AlbumentationsDataGenerator(
        folder_path=TRAIN_PATH,
        batch_size=BATCH_SIZE,
        img_size=IMG_SIZE,
        augment=CUSTOM_AUGMENTATION,
        oversample=True,
        min_per_class=400,
        backbone=backbone_name
    )

    val_gen = AlbumentationsDataGenerator(
        folder_path=VAL_PATH,
        batch_size=BATCH_SIZE,
        img_size=IMG_SIZE,
        shuffle=False,
        augment=None,
        oversample=False,
        backbone=backbone_name
    )

    # weights
    NUM_CLASSES = len(train_gen.classes)
    y_train = train_gen.get_labels()
    weights = compute_class_weight('balanced', classes=np.arange(NUM_CLASSES), y=y_train)
    weights = np.sqrt(weights)
    # Model
    model, backbone = build_model(backbone_name, input_shape=IMG_SIZE + (3,), num_classes=NUM_CLASSES)

    # Callbacks
    best_model_path = str(RESULTS_PATH / f"{backbone_name}_best_model.keras")
    callbacks = [
        ModelCheckpoint(best_model_path, save_best_only=True, monitor='val_f1_macro', mode='max'),
        EarlyStopping(monitor='val_f1_macro', patience=5, restore_best_weights=True, mode='max'),
        ReduceLROnPlateau(monitor='val_f1_macro', factor=0.5, patience=2, min_lr=1e-6, mode='max'),
        ClassMetricsCallback(val_gen, class_names=train_gen.classes)
    ]

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
        loss=weighted_categorical_crossentropy(weights),
        metrics=['accuracy', precision_macro, recall_macro, f1_macro]
    )

    # Train
    history = model.fit(
        train_gen,
        validation_data=val_gen,
        epochs=30,
        callbacks=callbacks
    )

    print(f"\n=== TRAINING COMPLETE FOR {backbone_name} ===\n")